# 06 --- Conflict Resolution

**CCA Pattern**: The coordinator resolves contradictions using deterministic strategies:
1. Source reliability ranking
2. Majority consensus
3. Flag for human review

This is programmatic enforcement -- not LLM judgment.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))
sys.path.insert(0, str(Path('.').resolve()))

In [ ]:
from research_agents.agent.conflict_resolver import (
    resolve_conflict, resolve_conflicts, RELIABILITY_SCORES
)
from research_agents.models.research import SourceReliability, ConflictRecord

## How the Conflict Resolver Works

The `conflict_resolver.py` module uses **deterministic strategies** -- no LLM reasoning at all. This is the CCA principle that programmatic enforcement beats prompt-based guidance.

### The ConflictRecord Model

```python
class ConflictRecord(BaseModel):
    claim: str                 # The disputed factual claim
    sources_for: list[str]     # URLs supporting the claim
    sources_against: list[str] # URLs contradicting the claim
    resolution: str            # "majority", "highest_reliability", "flagged_for_human"
    confidence: float          # 0.0 to 1.0
```

### Reliability Scoring

Sources are scored by reliability tier:

| Tier | Score | Examples |
|------|-------|---------|
| HIGH | 3 | `.gov`, peer-reviewed journals, official statistics |
| MEDIUM | 2 | Established news outlets, `.edu`, consultancies |
| LOW | 1 | Blogs, unverified sources |
| UNKNOWN | 0 | Not yet assessed |

### Three-Tier Resolution Strategy

1. **Reliability ranking**: Sum reliability scores for each side. Higher total wins. Confidence scales with the score difference.
2. **Majority consensus**: If reliability ties, the side with more sources wins. Confidence = majority_size / total.
3. **Human review**: If both reliability and count tie, flag for human review with low confidence (0.3).

## Strategy 1: Source Reliability Ranking

In [ ]:
# Show the scoring constants
print('Reliability scores:')
for tier, score in RELIABILITY_SCORES.items():
    print(f'  {tier.value:<10s} = {score}')
print()

# High-reliability source beats low-reliability
reliability = {
    'https://energy.gov/renewable-2024': SourceReliability.HIGH,
    'https://energyblog.example.com/renewables': SourceReliability.LOW,
}
record = resolve_conflict(
    claim='Renewable energy accounts for 30% of global electricity',
    sources_for=['https://energy.gov/renewable-2024'],
    sources_against=['https://energyblog.example.com/renewables'],
    reliability_lookup=reliability,
)
print(f'Claim: {record.claim}')
print(f'Resolution: {record.resolution}')
print(f'Confidence: {record.confidence:.2f}')

## Strategy 3: Equal Reliability + Equal Count -> Human Review

In [ ]:
# Equal reliability + equal count -> human review
reliability_equal = {
    'https://reuters.com': SourceReliability.MEDIUM,
    'https://bbc.com': SourceReliability.MEDIUM,
}
record2 = resolve_conflict(
    claim='Remote work productivity',
    sources_for=['https://reuters.com'],
    sources_against=['https://bbc.com'],
    reliability_lookup=reliability_equal,
)
print(f'Resolution: {record2.resolution}')
print(f'Confidence: {record2.confidence:.2f}')

### How This Connects to the ResearchReport

The `build_research_report()` function in `coordinator.py` uses conflict records to adjust the report's overall confidence score:

```python
base_confidence = 0.8
if gaps:
    base_confidence -= 0.1 * len(gaps)     # Each gap reduces confidence
if conflict_records:
    flagged = [c for c in conflict_records
               if c.resolution == "flagged_for_human"]
    base_confidence -= 0.05 * len(flagged)  # Unresolved conflicts reduce confidence
```

Reports that say 'we could not reach Source X' or 'these sources disagree and we flagged it for human review' are **more trustworthy** than reports that silently omit unavailable sources or pick the first result.

## CCA Exam Tip

> The conflict resolution question tests whether you understand that:
> - Source reliability ranking is the first strategy
> - Majority consensus applies when reliability is tied
> - Human review is the fallback, not auto-resolution
> - 'First result wins' is always the wrong answer
> - The resolution is **deterministic** -- same inputs always produce same output